In [ ]:
from pathlib import Path

from symexp import Model, LinExpr, BinVar, Var, Sense, VType
from symexp.solver import GurobiSolver

In [ ]:
lines = Path("assignments.csv").read_text().splitlines()
lines = [line.split(",") for line in lines[1:]]
id_map: dict[str, int] = {}
edges: list[tuple[int, int]] = []
edges_label: list[str] = []
for label, a, b in lines:
    a = a.strip()
    b = b.strip()
    if a not in id_map:
        id_map[a] = len(id_map)
    if b not in id_map:
        id_map[b] = len(id_map)
    edges.append((id_map[a], id_map[b]))
    edges_label.append(label)
id_imap: dict[int, str] = {v: k for k, v in id_map.items()}

In [ ]:
def setup_model(
    n_people: int,
    n_groups: int,
    edges: list[tuple[int, int]],
    group_diff: int = 5,
    paper_diff: int = 5,
) -> tuple[Model, list[list[BinVar]], list[LinExpr], list[LinExpr]]:
    """
    Returns
    - model
    - assignment_vars: (n_people, n_groups)
    - group_sizes: (n_groups,) number of people in each group
    - paper_sizes: (n_groups,) number of papers in each group
    """
    model = Model.create("Hello", LinExpr)
    assignment_vars: list[list[BinVar]] = []
    for i in range(n_people):
        vars_: list[BinVar] = []
        for j in range(n_groups):
            vars_.append(model.add_var(VType.BINARY, name=f"u_{i}_{j}"))
        # each person has exactly one assignment
        model.add_constraint(model.sum(vars_) == 1, name=f"u_{i}_sum")
        assignment_vars.append(vars_)

    # the number of papers in the same group
    cost = model.sum(
        [
            model.and_(assignment_vars[a][j], assignment_vars[b][j])
            for a, b in edges
            for j in range(n_groups)
        ]
    )

    # group size constraints
    group_sizes: list[LinExpr] = []
    for j in range(n_groups):
        group_sizes.append(model.sum(assignment_vars[i][j] for i in range(n_people)))

    min_group_size = model.min_(*group_sizes, name="min_group_size").value
    max_group_size = model.max_(*group_sizes, name="max_group_size").value
    model.add_constraint(max_group_size - min_group_size <= group_diff, name="max_group_differ")

    # paper size constraints
    paper_sizes: list[LinExpr] = []
    for j in range(n_groups):
        paper_sizes.append(
            model.sum(
                model.and_(assignment_vars[a][j], assignment_vars[b][j])
                for a, b in edges
            )
        )

    min_paper_size = model.min_(*paper_sizes, name="min_paper_size").value
    max_paper_size = model.max_(*paper_sizes, name="max_paper_size").value
    model.add_constraint(max_paper_size - min_paper_size <= paper_diff, name="max_paper_differ")

    model.set_objective(Sense.MAXIMIZE, cost)
    return model, assignment_vars, group_sizes, paper_sizes

In [ ]:
N_GROUPS1 = 4
model1, asgn1, group1, paper1 = setup_model(n_people=len(id_map), n_groups=N_GROUPS1, edges=edges)

In [ ]:
solver = GurobiSolver(model1) # , time_limit=180)
sol = solver.solve()
model1.set_solution(sol)

In [ ]:
# number of people in each group
print([int(g) for g in group1])
# number of papers in each group
print([int(p) for p in paper1])

In [ ]:
remaining_edges: list[tuple[int, int]] = []
for a, b in edges:
    assigned = False
    for j in range(N_GROUPS1):
        if int(asgn1[a][j]) and int(asgn1[b][j]):
            assigned = True
            break
    if not assigned:
        remaining_edges.append((a, b))
N_GROUPS2 = 2
model2, asgn2, group2, paper2 = setup_model(n_people=len(id_map), n_groups=N_GROUPS2, edges=remaining_edges)

In [ ]:
solver = GurobiSolver(model2)
sol = solver.solve()
model2.set_solution(sol)

In [ ]:
# number of people in each group
print([int(g) for g in group2])
# number of papers in each group
print([int(p) for p in paper2])

In [ ]:
# remaining papers
print(len(edges) - sum(map(int, paper1)) - sum(map(int, paper2)))

In [ ]:
# write output
paper_groups: list[str] = []
for (a, b), label in zip(edges, edges_label):
    assigned = N_GROUPS1 + N_GROUPS2
    for j in range(N_GROUPS2):
        if int(asgn2[a][j]) and int(asgn2[b][j]):
            assigned = j + N_GROUPS1
            break
    for j in range(N_GROUPS1):
        if int(asgn1[a][j]) and int(asgn1[b][j]):
            assigned = j
            break
    paper_groups.append(chr(ord('A') + assigned))
with Path("paper_rooms.csv").open("w") as f:
    f.write("paper_id,room\n")
    for paper, group in zip(edges_label, paper_groups):
        f.write(f"{paper},{group}\n")

with Path("people_rooms.csv").open("w") as f:
    f.write("email,room1,room2\n")
    for i in range(len(id_map)):
        group1 = None
        for j in range(N_GROUPS1):
            if int(asgn1[i][j]):
                group1 = j
                break
        assert group1 is not None
        group2 = None
        for j in range(N_GROUPS2):
            if int(asgn2[i][j]):
                group2 = j
                break
        assert group2 is not None
        g1 = chr(ord('A') + group1)
        g2 = chr(ord('A') + group2 + N_GROUPS1)
        f.write(f"{id_imap[i]},{g1},{g2}\n")